In [68]:
#! python -m pip install numpy scipy matplotlib
import scipy as scp
import numpy as np
import matplotlib.pyplot as plt
from collections import deque
import time
from IPython.display import clear_output

In [69]:
#csi_largo = np.random.randn(500, 64) + 1j * np.random.randn(500, 64)

In [70]:
csi_ex = np.array([
    # Paquete 1 (t = 0)
    [13.2 + 7.2j,  5.4 + 14.0j, -6.2 + 13.6j, -14.1 + 5.0j,
    -13.4 - 6.6j, -4.6 - 14.3j,  7.0 - 13.3j,  14.4 - 4.2j],
    
    # Paquete 2 (t = 1)
    [11.7 + 9.3j,  3.2 + 14.6j, -8.3 + 12.5j, -14.7 + 3.0j,
    -11.8 - 9.2j, -2.5 - 14.8j,  8.8 - 12.1j,  14.8 - 2.3j],
    
    # Paquete 3 (t = 2)
    [ 9.9 + 11.2j,  0.9 + 15.0j, -10.2 + 11.0j, -14.9 + 0.9j,
    -9.9 - 11.2j, -0.3 - 15.0j,  10.4 - 10.8j,  14.9 - 0.3j]
])

fases = []#lpm


In [71]:
#ventana deslizante para ir laburando con csi nuevo entonces me da rpm en tiempo real cambiante
tamaño_ventana = 500 # laburo con 500 paquetes
índice_renovación = 20 # cada cuántos se renueva
csi = deque(maxlen=tamaño_ventana) # deque saca del principio y agrega al final de la lista
paquetes = 0
while True:
    paquete_nuevo = np.random.randn(64) + 1j * np.random.randn(64)
    csi.append(paquete_nuevo)
    paquetes += 1
    time.sleep(0.01)
    if len(csi) < tamaño_ventana:
        if paquetes % 20 == 0:
            print(f" {len(csi)}/{tamaño_ventana} paquetes")
        continue

    if paquetes % índice_renovación == 0: # % calcula el resto, así que va a dar cero cuando sea 20
        csi_largo = np.array(csi)
        

 20/500 paquetes
 40/500 paquetes
 60/500 paquetes
 80/500 paquetes
 100/500 paquetes
 120/500 paquetes
 140/500 paquetes
 160/500 paquetes
 180/500 paquetes
 200/500 paquetes
 220/500 paquetes
 240/500 paquetes
 260/500 paquetes
 280/500 paquetes
 300/500 paquetes
 320/500 paquetes
 340/500 paquetes
 360/500 paquetes
 380/500 paquetes
 400/500 paquetes
 420/500 paquetes
 440/500 paquetes
 460/500 paquetes
 480/500 paquetes


KeyboardInterrupt: 

In [ ]:
for subportadora in csi_largo:
    fases_z =[]
    for z in subportadora:
        I = z.real  # parte real
        Q = z.imag # parte imaginaria 
        fase = np.arctan2(Q, I) 
        fases_z.append(fase)
        
    fases.append(fases_z)
fases_brutas = np.array(fases)
fases_unwrapped = np.unwrap(fases_brutas, axis=0) 
print(fases_unwrapped)

[[ -0.35836551   3.12397605   1.80014335 ...  -2.78369298  -0.23379441
   -0.56150598]
 [ -2.99548697   4.28071953   1.64588909 ...  -4.0860441   -3.28264608
   -1.74232813]
 [ -1.34867875   7.29741921   4.16704899 ...  -5.92043223  -4.86338899
   -2.69245236]
 ...
 [  1.49822261  54.57758212 -20.44144506 ... -58.4786803   12.32009784
   41.28808578]
 [  1.45483326  53.65218736 -19.62343966 ... -57.58638563   9.63379757
   38.18712795]
 [ -0.67417887  56.19557256 -20.3379201  ... -59.26590493   9.63252298
   38.64108359]]


In [ ]:
fs = 100 #asumo que la frecuencia de lo que me mande hardware será 100 paquetes x seg
nyquist = fs / 2 # tiene que ser la mitad de lo que recibe pq si no tosquea x alguna razon. se llama limite de nyquist
frecuencia_baja, frecuencia_alta = 0.1, 0.5 # 0.1 son 6rpm y 0.5 30rpm
low = frecuencia_baja / nyquist
high = frecuencia_alta / nyquist
b, a = scp.signal.butter(N=2, Wn=[low, high], btype='bandpass') #plantilla del filtro
fases_filtradas = scp.signal.filtfilt(b, a, fases_unwrapped, axis=0) #aplico el filtro a mi fase
print(fases_filtradas)


[[ 0.74840647 -8.59324349  7.70075506 ...  5.61559324 -9.18296271
  -4.7087487 ]
 [ 0.81621668 -8.38377866  7.93010018 ...  5.72983156 -9.13671357
  -4.79024406]
 [ 0.88436804 -8.17070288  8.15888302 ...  5.8432324  -9.08823969
  -4.871013  ]
 ...
 [-0.04152473 -0.05552986 -0.07342382 ...  0.07249253  0.05497861
   0.01338941]
 [-0.03462874 -0.04680032 -0.06208822 ...  0.06094023  0.04642838
   0.01193324]
 [-0.02853169 -0.03898274 -0.05187777 ...  0.05061198  0.03873982
   0.01048767]]


In [ ]:
varianzas = np.var(fases_filtradas, axis=0) #calcula varianza
mejor_subportadora = np.argmax(varianzas) #agarro la subportadora de + varianza
mejor_señal = fases_filtradas[:, mejor_subportadora]

# 2. FFT con Zero-Padding (n_fft = 10000 para dar resolución fina en Hz)
n_fft = 10000 
fft_valores = np.fft.fft(mejor_señal, n=n_fft)
magnitudes = np.abs(fft_valores)
frecuencias = np.fft.fftfreq(n_fft, d=1/fs)

# 3. Mapear a frecuencias positivas y a RPM
mitad = n_fft // 2
frecuencias_pos = frecuencias[:mitad]
magnitudes_pos = magnitudes[:mitad]
rpm_pos = frecuencias_pos * 60.0

# 4. Máscara booleana para el rango de respiración humana (6 a 30 RPM)
mascara_humana = (rpm_pos >= 6.0) & (rpm_pos <= 30.0)
rpm_validas = rpm_pos[mascara_humana]
magnitudes_validas = magnitudes_pos[mascara_humana]

# 5. Detección de presencia y cálculo de RPM
if len(magnitudes_validas) > 0:
    indice_pico = np.argmax(magnitudes_validas)
    pico_potencia = magnitudes_validas[indice_pico]
    promedio_ruido = np.mean(magnitudes_pos)
    
    # Criterio: el pico debe destacar sobre el ruido de fondo
    if pico_potencia > (3.0 * promedio_ruido):
        rpm_detectadas = rpm_validas[indice_pico]
        print(f"Presencia detectada: Ritmo: {rpm_detectadas:.1f} RPM (Subportadora {mejor_subportadora})")
    else:
        print("Sin presencia humana detectada ")
        #debería poner algo para ver que sea continuo y que se vaya mostrando el cambio constante. así se ve el rpm en cada momento y aparte si fue un ruido en el rango pero que ocurrio una vez lo saco


Presencia detectada: Ritmo: 10.2 RPM (Subportadora 5)


In [ ]:
#! python -m pip install numpy scipy matplotlib
from collections import deque
import time
from IPython.display import clear_output  # <--- Limpia el output de Jupyter en tiempo real
import numpy as np
import scipy as scp

tamaño_ventana = 500
índice_renovación = 20
csi = deque(maxlen=tamaño_ventana)  # Elimina del principio cuando supera los 500
paquetes = 0

print("Iniciando visualización en tiempo real del buffer...")

while True:
    # 1. GENERAR Y AGREGAR NUEVO PAQUETE AL FINAL
    paquete_nuevo = np.random.randn(64) + 1j * np.random.randn(64)
    csi.append(paquete_nuevo)
    paquetes += 1

    time.sleep(0.01)  # Simula 100 Hz reales

    # Llenar el buffer inicial
    if len(csi) < tamaño_ventana:
        if paquetes % 50 == 0:
            print(
                f"Llenando buffer inicial... {len(csi)}/{tamaño_ventana} paquetes."
            )
        continue

    # 2. EVALUACIÓN Y MOSTRADO EN TIEMPO REAL CADA 20 PAQUETES
    if paquetes % índice_renovación == 0:
        csi_largo = np.array(csi)

        # Limpiamos la pantalla para dar efecto de animación en tiempo real
        clear_output(wait=True)

        print(f"paquete n°: {paquetes}")
        print("Muestra del primer paquete en memoria (csi_largo[0, :3]):")
        print(f"  -> {csi_largo[0, :3]}")
        print("Muestra del ÚLTIMO paquete recién entrado (csi_largo[-1, :3]):")
        print(f"  -> {csi_largo[-1, :3]}")

        # --- TU PROCESAMIENTO MATEMÁTICO ---
        fases_brutas = np.angle(csi_largo)
        fases_unwrapped = np.unwrap(fases_brutas, axis=0)

        fs = 100
        nyquist = fs / 2
        low, high = 0.1 / nyquist, 0.5 / nyquist
        b, a = scp.signal.butter(N=2, Wn=[low, high], btype="bandpass")
        fases_filtradas = scp.signal.filtfilt(b, a, fases_unwrapped, axis=0)

        varianzas = np.var(fases_filtradas, axis=0)
        mejor_subportadora = np.argmax(varianzas)
        mejor_señal = fases_filtradas[:, mejor_subportadora]

        n_fft = 2048
        fft_valores = np.fft.fft(mejor_señal, n=n_fft)
        magnitudes = np.abs(fft_valores[: n_fft // 2])
        frecuencias = np.fft.fftfreq(n_fft, d=1 / fs)[: n_fft // 2]
        rpm_pos = frecuencias * 60.0

        mascara_humana = (rpm_pos >= 6.0) & (rpm_pos <= 30.0)
        rpm_validas = rpm_pos[mascara_humana]
        magnitudes_validas = magnitudes[mascara_humana]

        if len(magnitudes_validas) > 0:
            indice_pico = np.argmax(magnitudes_validas)
            pico_potencia = magnitudes_validas[indice_pico]
            promedio_ruido = np.mean(magnitudes)

            if pico_potencia > (3.0 * promedio_ruido):
                rpm_detectadas = rpm_validas[indice_pico]
                print(
                    f"🟢 PRESENCIA: {rpm_detectadas:.1f} RPM (Subportadora {mejor_subportadora})"
                )
            else:
                print("⚪ SIN PRESENCIA DETECTADA")

📦 PAQUETE RECIBIDO N°: 1780
📏 Forma de la matriz (Shape): (500, 64)
   (Observa como el largo SE MANTIENE FIJO en 500, sacando del inicio)
Muestra del primer paquete en memoria (csi_largo[0, :3]):
  -> [-0.47507619-0.0873745j   0.08204631-0.23781654j -2.19056249-0.46919651j]
Muestra del ÚLTIMO paquete recién entrado (csi_largo[-1, :3]):
  -> [-1.81129128-0.82916336j -1.34975186+0.38938928j -0.06316521+0.99815631j]
--------------------------------------------------
🟢 PRESENCIA: 11.7 RPM (Subportadora 28)


KeyboardInterrupt: 